# 02b — Preprocesado completo: White Balance + CLAHE + Denoising

Este es el preprocesado **real** que usa el pipeline. Tres pasos en este orden:

1. **White Balance (Gray World)**: corrige el tinte de iluminación.
2. **CLAHE** sobre canal L de Lab: iguala el contraste local.
3. **Filtro bilateral**: reduce ruido conservando bordes.

Sirve para defender en la memoria por qué se usan las tres técnicas, mostrando el efecto de cada una.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt

from src.preprocessing import (
    preprocess,
    gray_world_white_balance,
    apply_clahe,
    bilateral_denoise,
)
from src.utils.io_utils import load_image, list_images

DATA_ROOT = Path('../data/external/combined')
plt.rcParams['figure.dpi'] = 90

## 1. Pipeline paso a paso sobre una imagen

Mostramos la transformación en cascada: original → +WB → +CLAHE → +Denoise.

In [ ]:
category = 'Apple'  # cualquier categoría del dataset combinado
img = load_image(list_images(DATA_ROOT / category)[0])

step1 = gray_world_white_balance(img)
step2 = apply_clahe(step1)
step3 = bilateral_denoise(step2)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, im, title in zip(axes,
                         [img, step1, step2, step3],
                         ['Original', '+ White Balance', '+ CLAHE', '+ Denoise (final)']):
    ax.imshow(im); ax.set_title(title, fontsize=11); ax.axis('off')
plt.tight_layout(); plt.show()

## 2. Aislar el efecto de cada técnica

Cada paso por separado, partiendo siempre de la imagen original. Útil para ver qué aporta cada uno **sin la influencia de los otros dos**.

In [ ]:
only_wb = preprocess(img, apply_wb=True, apply_clahe_step=False, apply_denoise=False)
only_clahe = preprocess(img, apply_wb=False, apply_clahe_step=True, apply_denoise=False)
only_denoise = preprocess(img, apply_wb=False, apply_clahe_step=False, apply_denoise=True)
full = preprocess(img)

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, im, title in zip(axes,
                         [img, only_wb, only_clahe, only_denoise, full],
                         ['Original', 'Solo WB', 'Solo CLAHE', 'Solo Denoise', 'Pipeline completo']):
    ax.imshow(im); ax.set_title(title, fontsize=11); ax.axis('off')
plt.tight_layout(); plt.show()

## 3. Galería sobre 6 categorías

Vista comparativa antes/después del pipeline completo en distintos tipos de producto.

In [ ]:
sample_categories = ['Apple', 'CEREAL', 'JUICE', 'CHIPS', 'JAM', 'TEA']
samples = []
for c in sample_categories:
    cat_dir = DATA_ROOT / c
    if cat_dir.is_dir():
        imgs = list_images(cat_dir)
        if imgs:
            samples.append((c, imgs[0]))

fig, axes = plt.subplots(len(samples), 2, figsize=(9, 3 * len(samples)))
for i, (cat, path) in enumerate(samples):
    img = load_image(path)
    img_pre = preprocess(img)
    axes[i, 0].imshow(img); axes[i, 0].set_title(f'{cat} — original', fontsize=10); axes[i, 0].axis('off')
    axes[i, 1].imshow(img_pre); axes[i, 1].set_title(f'{cat} — pipeline completo', fontsize=10); axes[i, 1].axis('off')
plt.tight_layout(); plt.show()

## 4. Conclusiones para la memoria

- **White Balance** corrige el tinte de iluminación. Notable cuando la foto tiene focos cálidos (la imagen se ve menos amarillenta).
- **CLAHE** sube el contraste local: zonas oscuras se ven mejor, zonas claras no se queman.
- **Bilateral Denoise** elimina el grano de cámara conservando los bordes (importante porque la siguiente etapa, region proposal, depende de los bordes).

**Próximo paso:** `03_region_proposal.ipynb` para ver cómo se generan las cajas candidatas sobre estas imágenes preprocesadas.